In [1]:
import os
import glob
import cv2
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from google.colab import drive

from sklearn.utils import shuffle
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, Dense, Dropout, Activation,
    GlobalAveragePooling2D, BatchNormalization,
    MaxPooling2D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ================================================================
# 1. COLAB FILE SYSTEM SETUP
# ================================================================

# Mount Google Drive
drive.mount('/content/drive')

# Unzip the dataset from Drive to the local Colab disk for speed
ZIP_PATH = "/content/drive/MyDrive/RoadGuard.zip" 
LOCAL_DIR = "/content/dataset"

if os.path.exists(ZIP_PATH):
    print("Unzipping dataset to local runtime...")
    if os.path.exists(LOCAL_DIR): shutil.rmtree(LOCAL_DIR)
    !unzip -q "{ZIP_PATH}" -d "{LOCAL_DIR}"
    print("Unzip complete.")
else:
    print(f"ERROR: Could not find {ZIP_PATH} in your Google Drive.")

# Update paths to point to the local unzipped folder
# Note: Linux is case-sensitive! Ensure "ROADGUARD" matches your folder name.
DATASET_PATH = os.path.join(LOCAL_DIR, "RoadGuard")
IMG_SIZE = 100

TRAIN_POTHOLE = os.path.join(DATASET_PATH, "train_pothole")
TRAIN_NORMAL  = os.path.join(DATASET_PATH, "train_normal")
TEST_POTHOLE  = os.path.join(DATASET_PATH, "test", "pothole")
TEST_NORMAL   = os.path.join(DATASET_PATH, "test", "normal")

# Verify directories
required_dirs = [TRAIN_POTHOLE, TRAIN_NORMAL, TEST_POTHOLE, TEST_NORMAL]
for d in required_dirs:
    if not os.path.isdir(d):
        print(f"❌ Missing directory: {d}")
    else:
        print(f"✅ Found: {d}")

# ================================================================
# 2. UTILITIES & FEATURE EXTRACTION
# ================================================================

def load_images(folder_path):
    images = []
    # Using glob to find images with various extensions
    files = []
    for ext in ["jpg", "jpeg", "png", "JPG", "PNG"]:
        files.extend(glob.glob(os.path.join(folder_path, f"*.{ext}")))
    
    for img_path in files:
        img = cv2.imread(img_path, 0) # Grayscale
        if img is None: continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
    return np.asarray(images)

def extract_severity_features(images):
    features = []
    for img in images:
        mean_val  = np.mean(img)
        std_val   = np.std(img)
        edges     = cv2.Canny(img, 50, 150)
        edge_dens = np.sum(edges > 0) / edges.size
        lap_var   = cv2.Laplacian(img.astype(np.float64), cv2.CV_64F).var()
        dark_ratio = np.sum(img < 80) / img.size
        features.append([mean_val, std_val, edge_dens, lap_var, dark_ratio])
    return np.array(features)

def assign_severity_labels(images, scaler=None, km=None, cluster_to_severity=None):
    feats = extract_severity_features(images)
    if len(feats) == 0:
        raise ValueError("No images found for processing.")

    if scaler is None:
        scaler = StandardScaler()
        feats_scaled = scaler.fit_transform(feats)
        km = KMeans(n_clusters=3, random_state=42, n_init=10)
        cluster_ids = km.fit_predict(feats_scaled)
        
        # Rank clusters by severity (lower mean + higher edge/texture = severe)
        centres = km.cluster_centers_
        severity_score = -centres[:, 0] + centres[:, 2] + centres[:, 3] + centres[:, 4]
        rank_order = np.argsort(severity_score)
        cluster_to_severity = {rank_order[i]: i for i in range(3)}
    else:
        feats_scaled = scaler.transform(feats)
        cluster_ids = km.predict(feats_scaled)

    severity_labels = np.array([cluster_to_severity[c] for c in cluster_ids])
    return severity_labels, cluster_ids, scaler, km, cluster_to_severity

# ================================================================
# 3. MODEL ARCHITECTURES
# ================================================================

def build_cnn(num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(128, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        GlobalAveragePooling2D(),
        
        Dense(256, activation="relu"),
        Dropout(0.4),
        Dense(num_classes, activation="softmax")
    ])
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
    ]

# ================================================================
# 4. EXECUTION PIPELINE
# ================================================================

print("\n--- Loading Data ---")
p_train = load_images(TRAIN_POTHOLE)
n_train = load_images(TRAIN_NORMAL)
p_test  = load_images(TEST_POTHOLE)
n_test  = load_images(TEST_NORMAL)

print(f"Training: {len(p_train)} potholes, {len(n_train)} normal")
print(f"Testing: {len(p_test)} potholes, {len(n_test)} normal")

# STAGE 1: Binary Classification
print("\n--- Training Stage 1: Binary Detection ---")
X_tr_bin = np.concatenate([p_train, n_train]).reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y_tr_bin = to_categorical(np.concatenate([np.ones(len(p_train)), np.zeros(len(n_train))]), 2)

X_te_bin = np.concatenate([p_test, n_test]).reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y_te_bin = to_categorical(np.concatenate([np.ones(len(p_test)), np.zeros(len(n_test))]), 2)

X_tr_bin, y_tr_bin = shuffle(X_tr_bin, y_tr_bin, random_state=42)

binary_model = build_cnn(2)
binary_model.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
binary_model.fit(X_tr_bin, y_tr_bin, epochs=30, batch_size=32, validation_split=0.1, callbacks=get_callbacks())

# STAGE 2: Severity Classification
print("\n--- Training Stage 2: Severity Level ---")
sev_labels_tr, _, sev_scaler, sev_km, sev_map = assign_severity_labels(p_train)
X_tr_sev = p_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y_tr_sev = to_categorical(sev_labels_tr, 3)

sev_labels_te, _, _, _, _ = assign_severity_labels(p_test, sev_scaler, sev_km, sev_map)
X_te_sev = p_test.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y_te_sev = to_categorical(sev_labels_te, 3)

severity_model = build_cnn(3)
severity_model.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
severity_model.fit(X_tr_sev, y_tr_sev, epochs=30, batch_size=32, validation_split=0.15, callbacks=get_callbacks())

# ================================================================
# 5. SAVE MODELS TO DRIVE
# ================================================================
binary_model.save("binary_pothole_cnn.keras")
severity_model.save("severity_pothole_cnn.keras")

# Move them to Drive so you don't lose them when the session ends
shutil.copy("binary_pothole_cnn.keras", "/content/drive/MyDrive/binary_pothole_cnn.keras")
shutil.copy("severity_pothole_cnn.keras", "/content/drive/MyDrive/severity_pothole_cnn.keras")

print("\n✅ Training Complete! Models saved to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping dataset to local runtime...
Unzip complete.
✅ Found: /content/dataset/RoadGuard/train_pothole
✅ Found: /content/dataset/RoadGuard/train_normal
✅ Found: /content/dataset/RoadGuard/test/pothole
✅ Found: /content/dataset/RoadGuard/test/normal

--- Loading Data ---
Training: 100 potholes, 100 normal
Testing: 10 potholes, 10 normal

--- Training Stage 1: Binary Detection ---
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.5556 - loss: 0.7133 - val_accuracy: 0.4500 - val_loss: 0.6890 - learning_rate: 0.0010
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 796ms/step - accuracy: 0.7000 - loss: 0.5569 - val_accuracy: 0.5500 - val_loss: 0.6792 - learning_rate: 0.0010
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 924ms/step - accuracy: 0.7611 - loss: 0.5134 - val_accuracy: 0.5500 - val_loss: 0.6729 - learning_rate: 0.0010
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 918ms/step - accuracy: 0.7833 - loss: 0.5064 - val_accuracy: 0.5500 - val_loss: 0.6706 - learning_rate: 0.0010
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.7944 - loss: 0.4340 - val_accuracy: 0.9000 - val_loss: 0.6316 - learning_rate: 0.0010
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 812ms/step - accuracy: 0.7944 - loss: 0.4466 - val_accuracy: 0.6000 - val_loss: 0.6291 - learning_rate: 0.0010
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8222 - loss: 0.4409 - val_accuracy: 0.6000 - val_loss

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.5412 - loss: 0.9370 - val_accuracy: 0.6000 - val_loss: 1.0299 - learning_rate: 0.0010
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 827ms/step - accuracy: 0.8353 - loss: 0.5392 - val_accuracy: 0.6667 - val_loss: 1.0003 - learning_rate: 0.0010
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 856ms/step - accuracy: 0.8824 - loss: 0.3600 - val_accuracy: 0.3333 - val_loss: 1.0068 - learning_rate: 0.0010
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 919ms/step - accuracy: 0.8941 - loss: 0.3146 - val_accuracy: 0.3333 - val_loss: 1.1291 - learning_rate: 0.0010
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 961ms/step - accuracy: 0.8000 - loss: 0.4346 - val_accuracy: 0.3333 - val_loss: 1.2936 - learning_rate: 0.0010
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 760ms/step - accuracy: 0.9176 - loss: 0.2413 - val_accuracy: 0.3333 - val_loss: 1.5126 - learning_rate: 5.0000e-04
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 908ms/step - accuracy: 0.9176 - loss: 0.2542 - val_accuracy: 0.3333 - v

In [3]:
import os
from google.colab import files

# 1. Define the Colab-local save path (temporary storage)
COLAB_SAVE_PATH = "/content/models"
os.makedirs(COLAB_SAVE_PATH, exist_ok=True)

if "binary_model" not in globals() or "severity_model" not in globals():
    raise RuntimeError("Run your training cells first so models are available.")

# 2. Save models to the Colab server disk
binary_path = os.path.join(COLAB_SAVE_PATH, "binary_pothole_cnn.keras")
severity_path = os.path.join(COLAB_SAVE_PATH, "severity_pothole_cnn.keras")

binary_model.save(binary_path)
severity_model.save(severity_path)

print(f"Models saved to Colab instance at: {COLAB_SAVE_PATH}")

# 3. Trigger Download to your local Windows 'Downloads' folder
print("Starting downloads to your local machine...")
files.download(binary_path)
files.download(severity_path)

Models saved to Colab instance at: /content/models
Starting downloads to your local machine...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>